# EDA — NYC Yellow Taxi Fare Prediction

Questions this notebook answers before any modeling starts:

- What does the raw data look like (shape, dtypes, missing values, duplicates)?
- Which columns are only known *after* the trip ends (leakage risk)?
- What's wrong with the raw values (0-mile trips, $0 fares, bad passenger counts)?
- What actually correlates with `fare_amount`?


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", None)
DATA_PATH = "../data/yellow_tripdata_2025_with_zones.csv"


## 1. Load and look

In [ ]:
df = pd.read_csv(DATA_PATH)
print(df.shape)
df.head(3)


In [ ]:
df.info()


## 2. Missing values and duplicates

In [ ]:
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
pd.DataFrame({"missing": missing, "pct": missing_pct}).query("missing > 0").sort_values("pct", ascending=False)


In [ ]:
df.duplicated().sum()


`RatecodeID`, `passenger_count`, `congestion_surcharge` and `Airport_fee` are all missing the
same ~15% of rows — worth checking whether it's the same rows before deciding how to handle it.

In [ ]:
same_rows = df[["RatecodeID", "passenger_count", "congestion_surcharge", "Airport_fee"]].isnull().all(axis=1)
same_rows.sum(), df["RatecodeID"].isnull().sum()


## 3. Which columns are leakage risks?

Anything only known once the trip has ended (tip, tolls, total, surcharges) can't be used as a model
input — the model would be predicting the fare using numbers that include the fare.

In [ ]:
leakage_cols = [
    "tip_amount", "tolls_amount", "total_amount", "improvement_surcharge",
    "mta_tax", "extra", "congestion_surcharge", "cbd_congestion_fee", "Airport_fee",
]
df[leakage_cols + ["fare_amount"]].corr()["fare_amount"].sort_values(ascending=False)


`total_amount` correlates almost perfectly with `fare_amount` by construction — exactly the kind
of leakage that needs to be dropped before feature engineering, not caught after the model looks
suspiciously good.

## 4. Sanity-check the core numeric columns

In [ ]:
print("fare_amount <= 0:", (df["fare_amount"] <= 0).sum())
print("fare_amount > 250:", (df["fare_amount"] > 250).sum())
print("trip_distance == 0:", (df["trip_distance"] == 0).sum())
print("trip_distance > 150:", (df["trip_distance"] > 150).sum())


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(df["fare_amount"].clip(upper=100), bins=60, ax=axes[0])
axes[0].set_title("fare_amount (clipped at $100)")
sns.histplot(df["trip_distance"].clip(upper=30), bins=60, ax=axes[1])
axes[1].set_title("trip_distance (clipped at 30 miles)")
plt.tight_layout()
plt.show()


Both are right-skewed with a long tail of expensive/long trips and a cluster of
zero-or-near-zero values that look like cancelled rides or GPS failures rather than real trips —
these get filtered out in the feature engineering step, not here.

## 5. Duration, derived from pickup/dropoff timestamps

In [ ]:
pickup = pd.to_datetime(df["tpep_pickup_datetime"], errors="coerce")
dropoff = pd.to_datetime(df["tpep_dropoff_datetime"], errors="coerce")
duration_min = (dropoff - pickup).dt.total_seconds() / 60

print("negative duration:", (duration_min < 0).sum())
print("duration < 1 min:", (duration_min < 1).sum())
print("duration > 5 hrs:", (duration_min > 300).sum())


## 6. What correlates with fare, once leakage is removed?

In [ ]:
candidate_cols = ["trip_distance", "passenger_count"]
tmp = df[candidate_cols + ["fare_amount"]].copy()
tmp["trip_duration_min"] = duration_min
tmp["pickup_hour"] = pickup.dt.hour

tmp.corr()["fare_amount"].drop("fare_amount").sort_values(ascending=False)


`trip_distance` and `trip_duration_min` dominate, as expected for a metered fare. `pickup_hour`
and `passenger_count` barely correlate on their own — they may still matter combined with other
features (rush hour, airport routes), which is why feature engineering builds interactions rather
than feeding raw columns straight into a model.

## 7. Zones and boroughs

In [ ]:
df["PU_Borough"].value_counts(dropna=False)


In [ ]:
airport_ids = {1, 132, 138}  # EWR, JFK, LGA
is_airport = df["PULocationID"].isin(airport_ids) | df["DOLocationID"].isin(airport_ids)
df.loc[is_airport, "fare_amount"].median(), df.loc[~is_airport, "fare_amount"].median()


Airport trips have a much higher median fare than everything else — largely because JFK and
Newark use flat rates rather than the meter. That's a strong candidate feature.

## Summary

- Drop rows with non-positive/extreme `fare_amount`, `trip_distance`, and duration — these are
  data errors, not real trips.
- Drop `total_amount`, `tip_amount`, `tolls_amount` and other billing columns — they're only known
  after the trip and leak the target.
- `trip_distance` and `trip_duration_min` are the strongest raw predictors of fare.
- Airport trips and pickup hour look promising once turned into engineered flags, not used raw.
- ~15% of rows share the same missing `RatecodeID` / `passenger_count` / surcharge fields —
  handle with imputation rather than dropping, since it's a meaningful chunk of the data.

Feature engineering picks up from here in `src/features.py`.